# Step 1c: SINCA Python Web Scraper
This script replaces the R-based scraper to download the latest Air Quality (PM2.5 and PM10) data from the Chilean SINCA portal for 2024 and 2025.

In [1]:
import pandas as pd
import requests
import io
import os
import time
from datetime import datetime

sinca_csv_path = '../data/raw/Registros de Calidad del Aire/Reportes SINCA/Data/DatosEstacioneSINCA.csv'
# Added skiprows=1 to ignore 'sep=;' at the first line
df_estaciones = pd.read_csv(sinca_csv_path, sep=';', decimal='.', encoding='utf-8', skiprows=1)

# Filter for daily MP2.5 and MP10
df_descarga = df_estaciones[
    (df_estaciones['pollutant'].isin(['mp2.5', 'mp10'])) & 
    (df_estaciones['metrica'] == 'Diario')
].copy()

print(f"Found {len(df_descarga)} monitoring stations/pollutants to download.\n")

base_url = "https://sinca.mma.gob.cl/cgi-bin/APUB-MMA/apub.tsindico2.cgi"
date_from = "230101"
date_to = "251231"
all_data = []

print("Starting downloads...")
for idx, row in df_descarga.iterrows():
    params = {
        'outtype': 'xcl',
        'macro': row['macro'],
        'from': date_from,
        'to': date_to,
        'path': '/usr/airviro/data/CONAMA/',
        'lang': 'esp'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        
        # Pass names explicitly because the first row of response.text is 'FECHA (YYMMDD);HORA (HHMM);'
        # But data rows have 5 semicolons, meaning 6 columns
        df_conc = pd.read_csv(
            io.StringIO(response.text), 
            sep=';', 
            decimal=',', 
            names=['fecha', 'hora', 'validados', 'preliminares', 'noValidados', 'extra'],
            skiprows=1,
            na_values=['', 'NA', ' '],
            engine='python'
        )
        
        if not df_conc.empty:
            df_conc['valor'] = df_conc['validados'].fillna(df_conc['preliminares']).fillna(df_conc['noValidados'])
            df_conc = df_conc.dropna(subset=['valor'])
            
            if not df_conc.empty:
                df_conc['estacion'] = row['estacion']
                df_conc['region'] = row['region']
                df_conc['longitud'] = row['longitud']
                df_conc['latitud'] = row['latitud']
                df_conc['pollutant'] = row['pollutant']
                all_data.append(df_conc[['fecha', 'estacion', 'region', 'longitud', 'latitud', 'pollutant', 'valor']])
                
    except Exception as e:
        print(f"Error downloading {row['estacion']} - {row['pollutant']}: {e}")
    
    time.sleep(0.5) # Polite delay to avoid hammering the server

if all_data:
    df_final = pd.concat(all_data, ignore_index=True)
    df_final['fecha'] = df_final['fecha'].astype(str).str.replace('\\.0$', '', regex=True).str.zfill(6)
    df_final['fecha'] = pd.to_datetime(df_final['fecha'], format='%y%m%d', errors='coerce')
    df_final = df_final.dropna(subset=['fecha'])
    
    out_dir = '../data/processed/sinca_aire'
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'sinca_2023_2025_diario.parquet')
    df_final.to_parquet(out_path, index=False)
    print(f"\nSuccessfully scraped and saved {len(df_final)} daily air quality records to {out_path}!")
else:
    print("No data was downloaded.")

Found 273 monitoring stations/pollutants to download.

Starting downloads...



Successfully scraped and saved 183358 daily air quality records to ../data/processed/sinca_aire\sinca_2023_2025_diario.parquet!
